# GP10 — Clip / Intersection / Overlay

Πρόβλεψε → Εκτέλεσε → Έλεγξε → Ερμήνευσε. Εδώ οι spatial operations δημιουργούν νέες geometries.


## 1. Προετοιμασία

Το πρώτο κελί κατεβάζει μόνο τα απαιτούμενα GeoJSON όταν δεν υπάρχει έγκυρη cache.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json
candidates=(Path('downloads/python/data'),Path('data'),Path('../../data'))
DATA_ROOT=next((p for p in candidates if (p/'gp10').is_dir()),Path('data'))
DATA_ROOT.mkdir(parents=True,exist_ok=True)
DATA_URLS={'landuse.geojson':'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/landuse.geojson','protected_zones.geojson':'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/protected_zones.geojson','study_area.geojson':'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp10/study_area.geojson'}
REQUIRED={'landuse.geojson':{'landuse_id','landuse'},'protected_zones.geojson':{'zone_id','zone_name'},'study_area.geojson':{'area_id','name'}}
def valid_cached(path,required):
    if not path.exists() or path.stat().st_size==0:return False
    try:
        d=json.loads(path.read_text(encoding='utf-8'));return d.get('type')=='FeatureCollection' and all(f.get('geometry') is not None and required.issubset(f.get('properties',{})) for f in d.get('features',[]))
    except Exception:return False
for filename,url in DATA_URLS.items():
    target=DATA_ROOT/'gp10'/filename
    if valid_cached(target,REQUIRED[filename]):print(f'[OK] Ήδη διαθέσιμο: {target}');continue
    target.parent.mkdir(parents=True,exist_ok=True)
    try:
        with urlopen(url,timeout=30) as r:target.write_bytes(r.read())
        if not valid_cached(target,REQUIRED[filename]):raise RuntimeError('μη έγκυρη δομή ή πεδία')
        print(f'[OK] Κατέβηκε: {target}')
    except Exception as exc:
        if target.exists():target.unlink()
        raise RuntimeError(f'Αποτυχία προετοιμασίας: {url} — {exc}') from exc


## 2. Inputs και πρόβλεψη

Πρόβλεψε τι αλλάζει σε rows, geometry και attributes σε clip και overlay.


In [ ]:
import geopandas as gpd
root=DATA_ROOT/'gp10'
study=gpd.read_file(root/'study_area.geojson')
landuse=gpd.read_file(root/'landuse.geojson')
protected=gpd.read_file(root/'protected_zones.geojson')
assert len(study)==1 and len(landuse)==2 and len(protected)==1
assert all(f.geometry.geom_type.eq('Polygon').all() for f in (study,landuse,protected))
assert study.crs.to_epsg()==landuse.crs.to_epsg()==protected.crs.to_epsg()==4326
print(landuse.columns.tolist(),protected.columns.tolist(),study.columns.tolist())


## 3. CRS και study mask

Το EPSG:4326 είναι γεωγραφικό. Μετέτρεψε όλα τα layers σε EPSG:2100 πριν από metric area.


In [ ]:
target_crs='EPSG:2100'
study=study.to_crs(target_crs);landuse=landuse.to_crs(target_crs);protected=protected.to_crs(target_crs)
for frame in (study,landuse,protected):
    assert frame.crs.to_epsg()==2100 and frame.geometry.notna().all() and frame.geometry.is_valid.all() and (~frame.geometry.is_empty).all()


## 4. Guided clip

Το clip περιορίζει το landuse στη mask και κρατά κυρίως τα landuse attributes.


In [ ]:
clipped=gpd.clip(landuse,study)
clipped['area_m2']=clipped.geometry.area
assert len(clipped)>=1 and clipped.geometry.geom_type.isin(['Polygon','MultiPolygon']).all()
assert clipped.geometry.is_valid.all() and (~clipped.geometry.is_empty).all() and (clipped.area_m2>0).all()
print('rows before/after',len(landuse),len(clipped),'area',clipped.area_m2.sum())


## 5. Guided overlay intersection

Η intersection δημιουργεί νέα κοινή geometry και συνδυάζει attributes από clipped και protected.


In [ ]:
overlay=gpd.overlay(clipped,protected,how='intersection')
overlay['area_m2']=overlay.geometry.area
assert len(overlay)>=1 and overlay.geometry.geom_type.isin(['Polygon','MultiPolygon']).all()
assert overlay.geometry.is_valid.all() and (~overlay.geometry.is_empty).all() and (overlay.area_m2>0).all()
assert overlay.area_m2.sum()<=clipped.area_m2.sum()+1e-6
print(overlay[['landuse_id','landuse','zone_id','area_m2']])


## 6. Controlled QA και visual check

Συμπλήρωσε πραγματικούς ελέγχους rows/areas, attributes και γεωμετρικής σχέσης. Μικρές αριθμητικές διαφορές στα όρια χρειάζονται τεκμηριωμένη ανοχή, όχι απόλυτη ισότητα.


In [ ]:
# TODO: συμπλήρωσε ουσιαστικές εκφράσεις QA.
clip_rows_ok=None
overlay_within_inputs_ok=None
attributes_ok=None
if clip_rows_ok is None or overlay_within_inputs_ok is None or attributes_ok is None:raise NotImplementedError('Σύγκρινε rows/areas, difference με ανοχή και attributes')
assert clip_rows_ok and overlay_within_inputs_ok and attributes_ok
ax=study.boundary.plot(figsize=(7,5),color='0.4');clipped.plot(ax=ax,alpha=.25,color='green');protected.boundary.plot(ax=ax,color='red');overlay.plot(ax=ax,alpha=.6,color='orange')


## 7. Επικάλυψη και μοναδική κάλυψη

Τα L01 και L02 επικαλύπτονται. Το component sum, το union area και η διαφορά τους απαντούν διαφορετικά ερωτήματα.


In [ ]:
component_area=clipped.geometry.area.sum()
union_area=clipped.geometry.union_all().area
double_counted_area=component_area-union_area
assert component_area>=union_area-1e-6
overlay_union=overlay.geometry.union_all()
outside_area=overlay_union.difference(clipped.geometry.union_all()).area
# 1 m² είναι ανοχή για το συγκεκριμένο συνθετικό παράδειγμα, όχι καθολική προδιαγραφή.
assert outside_area<=1.0
print({'component_sum_m2':component_area,'union_m2':union_area,'overlap_m2':double_counted_area,'overlay_outside_m2':outside_area})


## 8. Export και round-trip QA

Έλεγξε και τα δύο exported layers: features, CRS, geometry, attributes, IDs, bounds και areas.


In [ ]:
output_path='gp10_results.gpkg'
clipped.to_file(output_path,layer='clipped_landuse',driver='GPKG')
overlay.to_file(output_path,layer='landuse_protected',driver='GPKG')
reloaded_clipped=gpd.read_file(output_path,layer='clipped_landuse')
reloaded_overlay=gpd.read_file(output_path,layer='landuse_protected')
for reloaded,original,required in ((reloaded_clipped,clipped,{'landuse_id','landuse','area_m2'}),(reloaded_overlay,overlay,{'landuse_id','landuse','zone_id','zone_name','area_m2'})):
    assert len(reloaded)==len(original) and reloaded.crs.to_epsg()==2100
    assert reloaded.geometry.geom_type.isin(['Polygon','MultiPolygon']).all() and reloaded.geometry.notna().all() and reloaded.geometry.is_valid.all() and (~reloaded.geometry.is_empty).all()
    assert required.issubset(reloaded.columns)
    assert ((reloaded.total_bounds-original.total_bounds)**2<=1e-6).all()
    assert abs(reloaded.geometry.area.sum()-original.geometry.area.sum())<=max(1e-3,original.geometry.area.sum()*1e-9)
assert set(reloaded_clipped['landuse_id'])==set(clipped['landuse_id'])
assert set(reloaded_overlay['landuse_id'])==set(overlay['landuse_id'])
print('round-trip QA passed for clipped_landuse and landuse_protected',output_path)


## 9. Ανεξάρτητο Mini Challenge

Ανάφερε την επιφάνεια κάθε χρήσης γης μέσα στην protected zone και εξέτασε διπλομέτρηση. Διάλεξε μόνος σου πράξεις, QA και ερμηνεία. Το επόμενο κελί είναι σκόπιμα ημιτελές.


In [ ]:
raise NotImplementedError('Συμπλήρωσε το ανεξάρτητο Mini Challenge')


## 10. Readiness checkpoint

Μπορείς να εξηγήσεις clip, spatial join, overlay, metric CRS, validity, bounds, component sum, union και round-trip QA;
